## **E-Commerce Data Engineering Lakehouse**

Training program: Modern Data Engineering for AI Systems — SDAIA Academy

Kaggle/UCI Online Retail + PySpark + Delta Lake + Data Quality Gate

Pipeline: Data Source → Ingestion → Spark → Data Quality → Quality Gate → Quarantine / Delta Lake → Trusted Data → Analytics

1.Install Libraries

In [58]:
!pip install -q pyspark delta-spark openpyxl

2.Imports

In [59]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta import configure_spark_with_delta_pip
import pandas as pd
import os, shutil, json, glob


3.Spark

In [60]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Ecommerce_Lakehouse") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

4. Data Ingestion

In [61]:
USE_SAMPLE_DATA = True

if USE_SAMPLE_DATA:
    sample_data = [
        ("536365", "85123A", "WHITE HANGING HEART T-LIGHT HOLDER", 6, "2010-12-01 08:26:00", 2.55, 17850.0, "United Kingdom"),
        ("C536379", "D", "Discount", -1, "2010-12-01 09:41:00", 27.50, 14527.0, "United Kingdom"),
        (None, "21730", "GLASS STAR FROSTED T-LIGHT HOLDER", 0, "2010-12-01 09:41:00", -5.00, None, "United Kingdom") # Bad record
    ]
    columns = ["InvoiceNo", "StockCode", "Description", "Quantity", "InvoiceDate", "UnitPrice", "CustomerID", "Country"]
    raw_df = spark.createDataFrame(sample_data, columns)
else:
    raw_df = spark.read.option("header", "true").option("inferSchema", "true").csv("/content/Online Retail.csv")

In [62]:
def read_input(path):
    ext = os.path.splitext(path.lower())[1]
    if ext in [".xlsx", ".xls"]:
        pdf = pd.read_excel(path)
        return spark.createDataFrame(pdf)
    elif ext == ".csv":
        return spark.read.option("header", True).option("inferSchema", True).csv(path)
    else:
        raise ValueError("Supported files: .csv, .xlsx, .xls")

df_raw = read_input(INPUT_PATH)
print("Rows:", df_raw.count())
print("Columns:", len(df_raw.columns))
df_raw.printSchema()
df_raw.show(10, truncate=False)

Rows: 12
Columns: 8
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

+---------+---------+----------------------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                       |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+---------+---------+----------------------------------+--------+-------------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER|6       |2010-12-01 08:26:00|2.55     |17850     |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN               |6       |2010-12-01 08:26:00|3.39     |17850     |United Kingdom|
|536365   |84406B   |CREAM CU

5. Standardize the Schema

In [63]:
from pyspark.sql.functions import col

#Completeness or Validity
valid_condition = (
    col("InvoiceNo").isNotNull() &
    col("StockCode").isNotNull() &
    col("Quantity").isNotNull() &
    col("UnitPrice").isNotNull() &
    col("CustomerID").isNotNull() &
    (col("UnitPrice") >= 0) &
    (col("Quantity") != 0)
)

silver_df = raw_df.filter(valid_condition).dropDuplicates()
quarantine_df = raw_df.filter(~valid_condition)

**6**. Delta Layer

In [64]:
# Bronze
raw_df.write.format("delta").mode("overwrite").save("/tmp/delta/bronze")

#Silver
silver_df.write.format("delta").mode("overwrite").save("/tmp/delta/silver")

#Quarantine
quarantine_df.write.format("delta").mode("overwrite").save("/tmp/delta/quarantine")

7. Data Quality Checks

In [65]:
def quality_report(data):
    # Completeness: critical fields only. CustomerID is not required because the source contains missing customer IDs.
    critical = ["InvoiceNo", "StockCode", "Quantity", "UnitPrice", "InvoiceDate", "Country"]
    completeness_fail = data.filter(
        sum([F.col(c).isNull().cast("int") for c in critical]) > 0
    )

    # Validity
    validity_fail = data.filter(
        F.col("UnitPrice").isNull() |
        (F.col("UnitPrice") < 0) |
        F.col("Quantity").isNull() |
        (F.col("Quantity") == 0)
    )

    # Uniqueness: exact transaction-line duplicates
    business_cols = ["InvoiceNo","StockCode","Description","Quantity","InvoiceDate","UnitPrice","CustomerID","Country"]
    duplicate_groups = (
        data.groupBy(*business_cols)
        .count()
        .filter(F.col("count") > 1)
    )

    # Business rule:
    # cancellation invoices start with C and may have negative quantity.
    # normal invoices should have positive quantity.
    business_rule_fail = data.filter(
        ((~F.upper(F.col("InvoiceNo")).startswith("C")) & (F.col("Quantity") <= 0)) |
        ((F.upper(F.col("InvoiceNo")).startswith("C")) & (F.col("Quantity") >= 0))
    )

    result = {
        "completeness_fail_rows": completeness_fail.count(),
        "validity_fail_rows": validity_fail.count(),
        "duplicate_groups": duplicate_groups.count(),
        "business_rule_fail_rows": business_rule_fail.count()
    }
    result["PASS"] = all(v == 0 for v in result.values())
    return result, completeness_fail, validity_fail, duplicate_groups, business_rule_fail

report, completeness_fail, validity_fail, duplicate_groups, business_rule_fail = quality_report(df)
report

{'completeness_fail_rows': 0,
 'validity_fail_rows': 0,
 'duplicate_groups': 0,
 'business_rule_fail_rows': 0,
 'PASS': True}

8. Quality Gate

In [76]:
quality_gate = report["PASS"]
print("QUALITY REPORT")
for k,v in report.items():
    print(f"{k}: {v}")

print("\nOVERALL QUALITY GATE:", "PASS" if quality_gate else "FAIL")

QUALITY REPORT
completeness_fail_rows: 0
validity_fail_rows: 0
duplicate_groups: 0
business_rule_fail_rows: 0
PASS: True

OVERALL QUALITY GATE: PASS


In [67]:
critical = ["InvoiceNo", "StockCode", "Quantity", "UnitPrice", "InvoiceDate", "Country"]

row_fail_condition = (
    (sum([F.col(c).isNull().cast("int") for c in critical]) > 0) |
    F.col("UnitPrice").isNull() |
    (F.col("UnitPrice") < 0) |
    F.col("Quantity").isNull() |
    (F.col("Quantity") == 0) |
    (((~F.upper(F.col("InvoiceNo")).startswith("C")) & (F.col("Quantity") <= 0))) |
    ((F.upper(F.col("InvoiceNo")).startswith("C")) & (F.col("Quantity") >= 0))
)

failed_rows = df.filter(row_fail_condition).dropDuplicates()
failed_count = failed_rows.count()

if failed_count > 0:
    failed_rows.write.format("delta").mode("overwrite").save(QUARANTINE_PATH)

print("Quarantine rows:", failed_count)

Quarantine rows: 0


9. Trusted/Silver Transformation

In [68]:
trusted_df = (
    df
    .dropDuplicates(["InvoiceNo","StockCode","Description","Quantity","InvoiceDate","UnitPrice","CustomerID","Country"])
    .filter(F.col("InvoiceNo").isNotNull())
    .filter(F.col("StockCode").isNotNull())
    .filter(F.col("Quantity").isNotNull())
    .filter(F.col("UnitPrice").isNotNull())
    .filter(F.col("InvoiceDate").isNotNull())
    .filter(F.col("Country").isNotNull())
    .filter(F.col("UnitPrice") >= 0)
    .filter(F.col("Quantity") != 0)
    .withColumn("Revenue", F.round(F.col("Quantity") * F.col("UnitPrice"), 2))
    .withColumn(
        "TransactionType",
        F.when(F.upper(F.col("InvoiceNo")).startswith("C"), F.lit("Cancellation"))
         .otherwise(F.lit("Sale"))
    )
    .withColumn("OrderDate", F.to_date("InvoiceDate"))
)

trusted_df.write.format("delta").mode("overwrite").save(SILVER_PATH)

silver_df = spark.read.format("delta").load(SILVER_PATH)
print("Silver rows:", silver_df.count())
silver_df.show(10, truncate=False)

Silver rows: 12
+---------+---------+----------------------------------+--------+-------------------+---------+----------+--------------+-------+---------------+----------+
|InvoiceNo|StockCode|Description                       |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |Revenue|TransactionType|OrderDate |
+---------+---------+----------------------------------+--------+-------------------+---------+----------+--------------+-------+---------------+----------+
|536367   |21730    |GLASS STAR FROSTED T-LIGHT HOLDER |24.0    |2010-12-01 08:34:00|4.25     |13047     |United Kingdom|102.0  |Sale           |2010-12-01|
|536372   |22632    |HAND WARMER RED POLKA DOT         |6.0     |2010-12-01 09:01:00|1.85     |17850     |United Kingdom|11.1   |Sale           |2010-12-01|
|536375   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER|6.0     |2010-12-01 09:10:00|2.55     |17850     |United Kingdom|15.3   |Sale           |2010-12-01|
|536366   |22632    |HAND WARMER RED POLKA

10. Analytics Output

In [69]:
sales_df = silver_df.filter(F.col("TransactionType") == "Sale")

kpis = sales_df.agg(
    F.countDistinct("InvoiceNo").alias("total_orders"),
    F.countDistinct("CustomerID").alias("unique_customers"),
    F.sum("Revenue").alias("total_revenue"),
    F.avg("Revenue").alias("average_line_revenue")
)

print("KPI Summary")
kpis.show(truncate=False)

KPI Summary
+------------+----------------+-------------+--------------------+
|total_orders|unique_customers|total_revenue|average_line_revenue|
+------------+----------------+-------------+--------------------+
|9           |5               |455.14       |41.376363636363635  |
+------------+----------------+-------------+--------------------+



In [70]:
revenue_by_country = (
    sales_df.groupBy("Country")
    .agg(
        F.round(F.sum("Revenue"), 2).alias("total_revenue"),
        F.countDistinct("InvoiceNo").alias("orders")
    )
    .orderBy(F.col("total_revenue").desc())
)

revenue_by_country.show(15, truncate=False)

+--------------+-------------+------+
|Country       |total_revenue|orders|
+--------------+-------------+------+
|United Kingdom|431.74       |8     |
|France        |23.4         |1     |
+--------------+-------------+------+



In [71]:
revenue_by_product = (
    sales_df.groupBy("StockCode", "Description")
    .agg(
        F.round(F.sum("Revenue"), 2).alias("total_revenue"),
        F.sum("Quantity").alias("units_sold")
    )
    .orderBy(F.col("total_revenue").desc())
)

revenue_by_product.show(15, truncate=False)

+---------+----------------------------------+-------------+----------+
|StockCode|Description                       |total_revenue|units_sold|
+---------+----------------------------------+-------------+----------+
|22086    |PAPER CHAIN KIT 50'S CHRISTMAS    |204.0        |80.0      |
|21730    |GLASS STAR FROSTED T-LIGHT HOLDER |102.0        |24.0      |
|85123A   |WHITE HANGING HEART T-LIGHT HOLDER|45.9         |18.0      |
|22629    |SPACEBOY LUNCH BOX                |23.4         |12.0      |
|22632    |HAND WARMER RED POLKA DOT         |22.2         |12.0      |
|84406B   |CREAM CUPID HEARTS COAT HANGER    |22.0         |8.0       |
|71053    |WHITE METAL LANTERN               |20.34        |6.0       |
|21733    |RED HANGING HEART T-LIGHT HOLDER  |15.3         |6.0       |
+---------+----------------------------------+-------------+----------+



In [72]:
monthly_revenue = (
    sales_df
    .withColumn("YearMonth", F.date_format("InvoiceDate", "yyyy-MM"))
    .groupBy("YearMonth")
    .agg(F.round(F.sum("Revenue"), 2).alias("revenue"))
    .orderBy("YearMonth")
)

monthly_revenue.show(20, truncate=False)

+---------+-------+
|YearMonth|revenue|
+---------+-------+
|2010-12  |455.14 |
+---------+-------+



11. Demonstrate PASS and FAIL Quality Cases

In [73]:
# PASS CASE
pass_rows = [
    ("P001","10001","Item A",2,"2026-01-01 10:00:00",10.0,"10001","Saudi Arabia"),
    ("P002","10002","Item B",1,"2026-01-01 10:05:00",20.0,"10002","Saudi Arabia"),
    ("CP003","10003","Item C",-1,"2026-01-01 10:10:00",5.0,"10003","Saudi Arabia")
]
pass_df = spark.createDataFrame(pass_rows, required_cols)

pass_report, *_ = quality_report(pass_df)
print("PASS CASE:", pass_report)

PASS CASE: {'completeness_fail_rows': 0, 'validity_fail_rows': 0, 'duplicate_groups': 0, 'business_rule_fail_rows': 0, 'PASS': True}


In [74]:
# FAIL CASE
fail_rows = [
    ("F001","10001","Item A",2,"2026-01-01 10:00:00",10.0,"10001","Saudi Arabia"),
    ("F001","10001","Item A",2,"2026-01-01 10:00:00",10.0,"10001","Saudi Arabia"), # duplicate
    ("F002","10002","Item B",-2,"2026-01-01 10:05:00",20.0,"10002","Saudi Arabia"), # invalid normal invoice
    ("F003","10003","Item C",1,"2026-01-01 10:10:00",-5.0,"10003","Saudi Arabia") # invalid price
]
fail_df = spark.createDataFrame(fail_rows, required_cols)

fail_report, *_ = quality_report(fail_df)
print("FAIL CASE:", fail_report)

FAIL CASE: {'completeness_fail_rows': 0, 'validity_fail_rows': 1, 'duplicate_groups': 1, 'business_rule_fail_rows': 1, 'PASS': False}


12. Verify Delta Tables

In [75]:
print("Bronze path:", BRONZE_PATH)
print("Silver path:", SILVER_PATH)
print("Quarantine path:", QUARANTINE_PATH)

print("\nDelta history for Silver:")
spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_PATH}`").show(truncate=False)

Bronze path: /content/ecommerce_lakehouse/bronze/online_retail
Silver path: /content/ecommerce_lakehouse/silver/online_retail
Quarantine path: /content/ecommerce_lakehouse/quarantine/online_retail

Delta history for Silver:
+-------+-----------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation|operationParameters                   |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                                                           |userMetadata|engineInfo                         |
+-------+-----------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+